# Combine Area 1 fivemin power data (May 2025 – Jul 2026)

Load `{month}_fivemin_power.csv` from `2025_5`–`2025_12` and `2026_1`–`2026_7`, keep `eu_1`–`eu_24`, drop all-zero rows, and combine into one dataframe.

In [1]:
import re
from pathlib import Path

import pandas as pd

base_dir = Path(r"c:/Users/nhphuong/Desktop/Solar")

MONTH_NAMES = {
    1: "january",
    2: "february",
    3: "march",
    4: "april",
    5: "may",
    6: "june",
    7: "july",
    8: "august",
    9: "september",
    10: "october",
    11: "november",
    12: "december",
}

# (year, month) pairs for folders 2025_5 .. 2025_12 and 2026_1 .. 2026_7
folder_specs = [(2025, m) for m in range(5, 13)] + [(2026, m) for m in range(1, 8)]

expected_eu_cols = [f"eu_{i}" for i in range(1, 25)]

In [2]:
def load_eu_power_csv(file_path: Path) -> pd.DataFrame:
    """Read a fivemin power CSV and return time + eu_1..eu_24 columns."""
    df = pd.read_csv(file_path)

    time_col = next((c for c in df.columns if c.lower().startswith("time")), None)
    if time_col is None:
        raise KeyError(f"No time column found in {file_path}")

    eu_map = {}
    for col in df.columns:
        match = re.match(r"^eu_(\d+)", col.strip().lower())
        if match:
            eu_idx = int(match.group(1))
            if 1 <= eu_idx <= 24 and f"eu_{eu_idx}" not in eu_map:
                eu_map[f"eu_{eu_idx}"] = col

    out_df = pd.DataFrame({"time": df[time_col]})
    for eu_col in expected_eu_cols:
        out_df[eu_col] = df[eu_map[eu_col]] if eu_col in eu_map else 0

    return out_df


def find_fivemin_power_file(year: int, month: int) -> Path | None:
    """Locate {month}_fivemin_power.csv under Area_1 (or Area1 fallback)."""
    folder = base_dir / f"{year}_{month}"
    month_name = MONTH_NAMES[month]
    filename = f"{month_name}_fivemin_power.csv"

    for area in ("Area_1", "Area1"):
        candidate = folder / area / filename
        if candidate.exists():
            return candidate
    return None

In [3]:
all_dfs = []

for year, month in folder_specs:
    file_path = find_fivemin_power_file(year, month)
    if file_path is None:
        print(f"Skip (not found): {year}_{month} -> {MONTH_NAMES[month]}_fivemin_power.csv")
        continue

    month_df = load_eu_power_csv(file_path)
    all_dfs.append(month_df)
    print(f"Loaded {file_path.relative_to(base_dir)}: {len(month_df):,} rows")

if not all_dfs:
    raise FileNotFoundError("No fivemin power files were found for the requested folders.")

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df["time"] = pd.to_datetime(combined_df["time"])
combined_df = combined_df.sort_values("time").reset_index(drop=True)

rows_before = len(combined_df)
combined_df = combined_df[(combined_df[expected_eu_cols] != 0).any(axis=1)].reset_index(drop=True)
rows_removed = rows_before - len(combined_df)

print(f"\nCombined: {len(combined_df):,} rows ({rows_removed:,} all-zero rows removed)")
print(f"Date range: {combined_df['time'].min()} -> {combined_df['time'].max()}")
combined_df.head()

Loaded 2025_5\Area_1\may_fivemin_power.csv: 8,928 rows
Loaded 2025_6\Area_1\june_fivemin_power.csv: 8,640 rows
Loaded 2025_7\Area_1\july_fivemin_power.csv: 8,928 rows
Skip (not found): 2025_8 -> august_fivemin_power.csv
Skip (not found): 2025_9 -> september_fivemin_power.csv
Loaded 2025_10\Area_1\october_fivemin_power.csv: 8,928 rows
Loaded 2025_11\Area_1\november_fivemin_power.csv: 8,652 rows
Loaded 2025_12\Area_1\december_fivemin_power.csv: 8,928 rows
Loaded 2026_1\Area_1\january_fivemin_power.csv: 8,928 rows
Loaded 2026_2\Area_1\february_fivemin_power.csv: 8,064 rows
Skip (not found): 2026_3 -> march_fivemin_power.csv
Skip (not found): 2026_4 -> april_fivemin_power.csv
Loaded 2026_5\Area1\may_fivemin_power.csv: 8,928 rows
Loaded 2026_6\Area1\june_fivemin_power.csv: 8,640 rows
Loaded 2026_7\Area1\july_fivemin_power.csv: 7,200 rows

Combined: 51,005 rows (43,759 all-zero rows removed)
Date range: 2025-05-01 06:15:00 -> 2026-07-25 21:10:00


,time,eu_1,eu_2,eu_3,eu_4,eu_5,eu_6,eu_7,eu_8,eu_9,...,eu_15,eu_16,eu_17,eu_18,eu_19,eu_20,eu_21,eu_22,eu_23,eu_24
0,2025-05-01 06:15:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0303,0.0000,0.0000,...,0.0,0.0000,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000
1,2025-05-01 06:20:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0387,0.0000,0.0000,...,0.0,0.0000,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000
2,2025-05-01 06:25:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0585,0.0000,0.0000,...,0.0,0.0000,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000
3,2025-05-01 06:35:00,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0350,5.3213,0.0000,...,0.0,0.0000,0.000,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.0000
4,2025-05-01 06:40:00,16.8315,5.6704,28.2925,39.8013,5.6281,0.0,0.0798,59.4130,17.0214,...,0.0,29.7065,5.707,41.7165,0.0,5.6572,5.9691,5.6188,0.2228,5.6476


In [4]:
output_csv_path = base_dir / "all_data" / "combined_may2025_jul2026_area1.csv"
combined_df.to_csv(output_csv_path, index=False)
print(f"Saved: {output_csv_path}")

Saved: c:\Users\nhphuong\Desktop\Solar\all_data\combined_may2025_jul2026_area1.csv
